In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("AmazonReviews").master("local[*]").config("spark.sql.caseSensitive", "true").getOrCreate()

df_meta_beauty = spark.read.json("meta_All_Beauty.jsonl")
df_meta_beauty.columns


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/01 14:35:21 WARN Utils: Your hostname, Pangkaews-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.32.158.224 instead (on interface en0)
26/03/01 14:35:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 14:35:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

['average_rating',
 'bought_together',
 'categories',
 'description',
 'details',
 'features',
 'images',
 'main_category',
 'parent_asin',
 'price',
 'rating_number',
 'store',
 'title',
 'videos']

In [2]:
from pyspark.sql.functions import count

df_meta_beauty.groupBy("bought_together").agg(count("*")).show()
#since all row in bought_together is null, we decide to drop it in next step

26/03/01 14:35:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------+--------+
|bought_together|count(1)|
+---------------+--------+
|           NULL|  112590|
+---------------+--------+



In [3]:
# change data type and drop unneccessary column
# drop images, videos
from pyspark.sql.functions import col, when, size
from pyspark.sql.types import DoubleType
from pyspark.sql import functions as F

df_meta_beauty = df_meta_beauty.drop("images","videos","bought_together")



In [4]:
# cleaned details column

from pyspark.sql.types import StructType, StructField, StringType, MapType
from pyspark.sql import functions as F


# flat array data
df_flat = df_meta_beauty.select("*", "details.*").drop("details")
total_rows = df_flat.count()

# calculate how many null in each column
# we use f"`{c}`" to avoid error which can cause by "." or spaces
null_counts = df_flat.select([F.count(F.when(F.col(f"`{c}`").isNull(), c)).alias(c) for c in df_flat.columns]).collect()[0].asDict()

# identify column which has less missing value than threshold (80%)
# this value can be changed
cols_to_keep = [
    col for col, missing_count in null_counts.items() 
    if (missing_count / total_rows) < 0.80
]

# select columns which satisfy our threshold
df_final = df_flat.select([F.col(f"`{c}`") for c in cols_to_keep])

print(f"Success! Kept {len(cols_to_keep)} columns.")


df_final = df_final.drop("UPC","Is Discontinued By Manufacturer","Package Dimensions","Product Dimensions","Unit Count")
df_final.columns


[Stage 10:=====>                                                   (1 + 9) / 10]

Success! Kept 19 columns.


['average_rating',
 'categories',
 'description',
 'features',
 'main_category',
 'parent_asin',
 'rating_number',
 'store',
 'title',
 'Age Range (Description)',
 'Brand',
 'Hair Type',
 'Item Form',
 'Material']

In [5]:
from pyspark.sql import functions as F

#handle features columns

# turn to string
df_ready = df_final.withColumn("features_string", F.concat_ws(", ", F.col("features")))

# count row for calculate percentage of missing value
total_rows = df_ready.count()

# calculate null
null_counts = df_ready.select([F.count(F.when(F.col(f"`{c}`").isNull(), c)).alias(c) for c in df_ready.columns]).collect()[0].asDict()

# select only column which has missing value less than 80%
cols_to_keep = [col for col, count in null_counts.items() if (count / total_rows) < 0.80]
df_final_clean = df_ready.select([F.col(f"`{c}`") for c in cols_to_keep])

print(f"Final columns: {df_final_clean.columns}")

Final columns: ['average_rating', 'categories', 'description', 'features', 'main_category', 'parent_asin', 'rating_number', 'store', 'title', 'Age Range (Description)', 'Brand', 'Hair Type', 'Item Form', 'Material', 'features_string']


In [11]:
# cleaning pipeline

from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, StringType
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover
import re
import nltk
from nltk.stem import WordNetLemmatizer
import emoji
from pyspark.sql import functions as F

# download NLTK
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

# function to convert emoji to their meaning
def emoji_to_text(text):
    if not text:
        return ""
    return emoji.demojize(text, delimiters=(" ", " "))

# change 2 or more repeated character to 1 repeated character
def normalize_elongated(word):
    if not word:
        return ""
    return re.sub(r'(.)\1{2,}', r'\1\1', word)
    
# lemmatisation
def lemmatize_words(words):
    if not words:
        return []
    return [lemmatizer.lemmatize(w) for w in words]

# convert python function into UDF, so spark can use it
normalize_elongated_udf = F.udf(lambda words: [normalize_elongated(w) for w in words], ArrayType(StringType()))
emoji_to_text_udf = F.udf(emoji_to_text, StringType())
lemmatize_udf = F.udf(lemmatize_words, ArrayType(StringType()))


def preprocessing(df, text_col):
    # convert data to string
    df_cleaned = df.withColumn(text_col, F.coalesce(col(text_col).cast("string"), F.lit("")))
    
    # remove html and links
    df_cleaned = df_cleaned.withColumn(text_col, F.regexp_replace(col(text_col), r"http\S+|www\S+|<.*?>", " "))

    # convert emoji to their meaning
    df_cleaned = df_cleaned.withColumn(text_col, emoji_to_text_udf(col(text_col)))

    # lower case
    df_cleaned = df_cleaned.withColumn(text_col, F.lower(col(text_col)))

    # tokenisation
    tokenizer = RegexTokenizer(inputCol=text_col, outputCol=f"{text_col}_words", pattern="\\W")
    df_cleaned = tokenizer.transform(df_cleaned)

    # elongated word like soooo good -> so good
    df_cleaned = df_cleaned.withColumn(f"{text_col}_words", normalize_elongated_udf(col(f"{text_col}_words")))

    # remove stop word
    stopwords_remover = StopWordsRemover(inputCol=f"{text_col}_words", outputCol=f"{text_col}_no_stop")
    df_cleaned = stopwords_remover.transform(df_cleaned)
    
    #lemmatisation
    df_cleaned = df_cleaned.withColumn(f"{text_col}_lemma", lemmatize_udf(col(f"{text_col}_no_stop")))
    
    # join back to strings
    df_cleaned = df_cleaned.withColumn(f"{text_col}_final", F.concat_ws(" ", col(f"{text_col}_lemma")))
    
    # drop unneccesary columns
    df_cleaned = df_cleaned.drop(text_col, f"{text_col}_lemma", f"{text_col}_words", f"{text_col}_no_stop")
    df_cleaned = df_cleaned.withColumnRenamed(f"{text_col}_final", text_col)
    
    return df_cleaned


df_cleaned_descrip = preprocessing(df_final_clean, text_col="description")
df_cleaned_title = preprocessing(df_cleaned_descrip, text_col="title")
df_cleaned_features = preprocessing(df_cleaned_title, text_col="features_string")

df_cleaned_final = df_cleaned_features.drop("features")

# verify result
df_cleaned_final .select("description","title").show(5, truncate=False)


# save file
df_cleaned_final.coalesce(1).write.mode("overwrite").json("cleaned_meta_beauty")

[nltk_data] Downloading package wordnet to /Users/prim/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/prim/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
26/03/01 14:56:59 WARN StopWordsRemover: Default locale set was [en_TH]; however, it was not found in available locales in JVM, falling back to en_US locale. Set param `locale` in order to respect another locale.
26/03/01 14:56:59 WARN StopWordsRemover: Default locale set was [en_TH]; however, it was not found in available locales in JVM, falling back to en_US locale. Set param `locale` in order to respect another locale.
26/03/01 14:56:59 WARN StopWordsRemover: Default locale set was [en_TH]; however, it was not found in available locales in JVM, falling back to en_US locale. Set param `locale` in order to respect another locale.
                                                                                

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+
|description                                                                                                                                                                                                                                                                                                                                |title                                                                                                                                             |
+-------------------------------------